In [1]:
!pip install pandas numpy matplotlib scikit-learn tqdm

# UNIVERSAL Forecast Verification (Backtesting All Components)
For each component, we train on the first 80% of its timeline, and predict the final 20% to calculate aggregated accuracy.

In [2]:
import pandas as pd
import numpy as np
import glob
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, accuracy_score, recall_score
from tqdm import tqdm

file_paths = sorted(glob.glob('/Users/anshul/Documents/AI/data/raw/MOSFET/MOSFET_Device_*.csv'))
print(f"Found {len(file_paths)} device files.")

all_actuals = []
all_preds = []
all_actual_flags = []
all_pred_flags = []

for file_path in tqdm(file_paths, desc="Backtesting Devices"):
    try:
        df = pd.read_csv(file_path, usecols=['Device_ID', 'Aging_Hours', 'RDS_on_Ohm'])
        df = df.dropna(subset=['Aging_Hours', 'RDS_on_Ohm']).sort_values('Aging_Hours')
        
        if len(df) < 10:
            continue
            
        # 80-20 Time Split
        cutoff_hour = df['Aging_Hours'].quantile(0.8)
        train_df = df[df['Aging_Hours'] <= cutoff_hour]
        test_df = df[df['Aging_Hours'] > cutoff_hour]
        
        if len(test_df) == 0 or len(train_df) == 0:
            continue
            
        model = LinearRegression()
        model.fit(train_df[['Aging_Hours']], train_df['RDS_on_Ohm'])
        
        preds = model.predict(test_df[['Aging_Hours']])
        actuals = test_df['RDS_on_Ohm'].values
        
        # We define a dynamic anomaly threshold for backtesting as 5% over the max training value
        threshold = train_df['RDS_on_Ohm'].max() * 1.05
        
        all_actuals.extend(actuals)
        all_preds.extend(preds)
        
        all_actual_flags.extend((actuals > threshold).astype(int))
        all_pred_flags.extend((preds > threshold).astype(int))
        
    except Exception as e:
        pass

mae = mean_absolute_error(all_actuals, all_preds)
acc = accuracy_score(all_actual_flags, all_pred_flags)
rec = recall_score(all_actual_flags, all_pred_flags, zero_division=0)

print(f"\n--- AGGREGATED UNIVERSAL BACKTESTING RESULTS ---")
print(f"Total Future Datapoints Evaluated: {len(all_actuals)}")
print(f"Global Mean Absolute Error (MAE): {mae:.4f} Ohm")
print(f"Global Accuracy: {acc*100:.2f}%")
print(f"Global Recall:   {rec*100:.2f}% (Catching anomalies before they happen)")


Found 42 device files.


Backtesting Devices: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 42/42 [00:01<00:00, 39.39it/s]


--- AGGREGATED UNIVERSAL BACKTESTING RESULTS ---
Total Future Datapoints Evaluated: 220562
Global Mean Absolute Error (MAE): 3.5160 Ohm
Global Accuracy: 94.66%
Global Recall:   2.80% (Catching anomalies before they happen)
